# Praktikum Machine Learning: Naive Bayes untuk Deteksi SMS Spam
Notebook ini dirancang untuk dijalankan di **Google Colab**.

### Studi Kasus
Kita sering menerima SMS seperti:
- *"Selamat! Nomor Anda menang hadiah 100 juta, klaim sekarang!"* → **SPAM** (penipuan/promosi)
- *"Nanti sore jadi kerja kelompok di perpustakaan?"* → **BUKAN SPAM** (pesan normal)

Kita akan membuat model **Naive Bayes** yang bisa **menebak sendiri** apakah sebuah SMS termasuk spam atau bukan, dengan belajar dari contoh-contoh SMS.

### Ingat Kembali: Cara Kerja Naive Bayes
$$\text{Skor kelas} = P(\text{Kelas}) \times P(\text{kata}_1 \mid \text{Kelas}) \times P(\text{kata}_2 \mid \text{Kelas}) \times \dots$$
Model menghitung skor untuk kelas **spam** dan **bukan spam**, lalu memilih yang paling besar.

Penjelasan lengkap ada di file `naive_bayes.md`.

### Alur Praktikum
1. Import library
2. Siapkan data (Data)
3. Lihat dan pahami data
4. Preprocessing teks (Feature Engineering)
5. Bagi data latih dan data uji
6. Ubah teks jadi angka (Feature Engineering)
7. Latih model Naive Bayes (Algoritma)
8. Evaluasi model
9. Lihat apa yang dipelajari model
10. Coba dengan SMS baru
11. Eksperimen hyperparameter `alpha`

## 1. Import Library
Semua library di bawah ini sudah terpasang di Google Colab, jadi tidak perlu `pip install`.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Semua library berhasil diimpor!")

## 2. Menyiapkan Data
Kita membuat sendiri kumpulan SMS bahasa Indonesia beserta **label**-nya:
- `spam` → SMS penipuan / promosi
- `bukan spam` → SMS normal sehari-hari

Dalam ML, ini disebut **data berlabel** (*supervised learning*): model belajar dari contoh yang jawabannya sudah diketahui.

In [ ]:
sms_spam = [
    "selamat anda menang hadiah 100 juta klaim sekarang",
    "gratis pulsa 50 ribu untuk anda klik link ini",
    "promo spesial diskon 90 persen hanya hari ini",
    "anda terpilih menjadi pemenang undian berhadiah mobil",
    "klaim hadiah anda sekarang hubungi nomor ini",
    "dapatkan pinjaman cepat tanpa jaminan cair hari ini",
    "selamat nomor anda menang undian berhadiah motor",
    "bonus pulsa gratis untuk pelanggan setia klik link",
    "promo kuota internet murah diskon besar hari ini",
    "pinjaman dana cepat bunga rendah tanpa jaminan",
    "anda mendapat hadiah uang tunai segera transfer biaya admin",
    "menangkan iphone gratis dengan mengikuti undian ini",
    "diskon besar besaran semua produk promo terbatas",
    "kirim pulsa 20 ribu untuk klaim hadiah anda",
    "selamat akun anda terpilih mendapat bonus saldo gratis",
    "pinjaman online cair dalam 5 menit tanpa ribet",
    "hadiah langsung untuk anda klik link sekarang juga",
    "promo gratis ongkir dan cashback hanya hari ini",
    "anda pemenang undian berhadiah uang 50 juta",
    "dapatkan bonus kuota gratis dengan klik link berikut",
    "segera klaim voucher belanja gratis sebelum hangus",
    "investasi untung besar modal kecil hubungi kami",
    "transfer biaya admin untuk mencairkan hadiah anda",
    "kesempatan terakhir menang undian hadiah mobil mewah",
    "gratis voucher diskon untuk pelanggan terpilih",
]

sms_normal = [
    "nanti sore jadi kerja kelompok di perpustakaan",
    "jangan lupa besok ada kuis machine learning",
    "aku sudah sampai di kampus kamu dimana",
    "tolong kirim file tugas kemarin ya",
    "besok rapat organisasi jam tujuh pagi",
    "makan siang bareng di kantin yuk",
    "ibu minta tolong belikan beras pulang kuliah",
    "dosen bilang kelas hari ini diganti minggu depan",
    "kamu sudah mengerjakan tugas statistika belum",
    "nanti malam aku telepon ya",
    "jadwal ujian sudah keluar cek di grup kelas",
    "terima kasih sudah membantu tugas kemarin",
    "ayah sudah sampai rumah dengan selamat",
    "besok pagi aku jemput di depan kos ya",
    "praktikum python dipindah ke laboratorium dua",
    "selamat ulang tahun semoga sehat selalu",
    "aku pinjam catatan kuliah kamu boleh",
    "rapat dibatalkan karena dosen ada acara",
    "kapan kita mulai mengerjakan laporan praktikum",
    "jangan lupa bawa laptop untuk presentasi besok",
    "kamu pulang kampung kapan liburan nanti",
    "grup belajar kita pindah ke hari kamis",
    "aku sudah kirim tugas lewat email dosen",
    "sore ini hujan jangan lupa bawa payung",
    "mau titip makanan dari kantin tidak",
]

df = pd.DataFrame({
    "sms":   sms_spam + sms_normal,
    "label": ["spam"] * len(sms_spam) + ["bukan spam"] * len(sms_normal),
})

# Acak urutan baris agar spam dan bukan spam tercampur
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total data: {len(df)} SMS")
df.head(10)

## 3. Melihat dan Memahami Data
Sebelum melatih model, kita perlu tahu isi datanya: berapa jumlah tiap kelas, dan kata apa yang sering muncul.

In [ ]:
# Jumlah SMS per kelas
print(df["label"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label", hue="label", palette={"spam": "#e74c3c", "bukan spam": "#2ecc71"}, legend=False)
plt.title("Jumlah SMS per Kelas", fontweight="bold")
plt.xlabel("Kelas")
plt.ylabel("Jumlah SMS")
plt.show()

In [ ]:
# Kata yang paling sering muncul di setiap kelas
def kata_terbanyak(teks_list, n=10):
    semua_kata = " ".join(teks_list).split()
    return pd.Series(semua_kata).value_counts().head(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kata_terbanyak(df[df["label"] == "spam"]["sms"]).sort_values().plot.barh(ax=axes[0], color="#e74c3c")
axes[0].set_title("10 Kata Terbanyak di SMS SPAM", fontweight="bold")

kata_terbanyak(df[df["label"] == "bukan spam"]["sms"]).sort_values().plot.barh(ax=axes[1], color="#2ecc71")
axes[1].set_title("10 Kata Terbanyak di SMS BUKAN SPAM", fontweight="bold")

plt.tight_layout()
plt.show()

**Perhatikan:** kata seperti `hadiah`, `gratis`, `klaim`, `undian` banyak muncul di spam, sedangkan `kuliah`, `tugas`, `besok` banyak muncul di SMS normal.

Pola inilah yang nanti dipelajari Naive Bayes dalam bentuk **peluang**.

## 4. Preprocessing Teks
Membersihkan teks supaya seragam:
1. Huruf kecil semua (*case folding*)
2. Hapus angka dan tanda baca
3. Hapus spasi berlebih

In [ ]:
def bersihkan_teks(teks):
    teks = teks.lower()                       # 1. huruf kecil
    teks = re.sub(r"[^a-z\s]", " ", teks)     # 2. buang angka & tanda baca
    teks = re.sub(r"\s+", " ", teks).strip()  # 3. rapikan spasi
    return teks

df["sms_bersih"] = df["sms"].apply(bersihkan_teks)

print("Contoh sebelum :", df["sms"][0])
print("Contoh sesudah :", df["sms_bersih"][0])

## 5. Membagi Data Latih dan Data Uji
- **Data latih (train)**: dipakai model untuk **belajar** (75%)
- **Data uji (test)**: disembunyikan dari model, dipakai untuk **menguji** apakah model benar-benar paham (25%)

Analogi: siswa belajar dari soal latihan, lalu diuji dengan soal ujian yang **belum pernah dilihat**. Kalau ujiannya memakai soal latihan yang sama, nilainya tidak jujur.

`stratify` menjaga agar proporsi spam dan bukan spam di data latih dan data uji tetap seimbang.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["sms_bersih"], df["label"],
    test_size=0.25,
    random_state=42,
    stratify=df["label"],
)

print(f"Data latih : {len(X_train)} SMS")
print(f"Data uji   : {len(X_test)} SMS")

## 6. Mengubah Teks Menjadi Angka (Bag of Words)
Model tidak bisa membaca huruf, jadi teks diubah menjadi **jumlah kemunculan tiap kata** dengan `CountVectorizer`.

Contoh:

| SMS | gratis | hadiah | tugas | ... |
|-----|--------|--------|-------|-----|
| "gratis hadiah gratis" | 2 | 1 | 0 | ... |
| "kirim tugas" | 0 | 0 | 1 | ... |

**Penting:** `fit_transform` hanya untuk data latih (supaya kosakata dipelajari dari data latih saja). Data uji cukup `transform`.

In [ ]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)   # belajar kosakata + ubah ke angka
X_test_vec  = vectorizer.transform(X_test)        # hanya ubah ke angka

print(f"Jumlah kosakata (fitur): {len(vectorizer.get_feature_names_out())} kata")
print(f"Bentuk data latih: {X_train_vec.shape}  → (jumlah SMS, jumlah kata)")

# Intip sebagian matriksnya
pd.DataFrame(
    X_train_vec.toarray()[:5],
    columns=vectorizer.get_feature_names_out()
).iloc[:, :15]

## 7. Melatih Model Naive Bayes
Kita memakai **Multinomial Naive Bayes**, jenis yang paling cocok untuk data **jumlah kata**.

- `alpha=1.0` → **Laplace smoothing** (+1), supaya kata yang belum pernah terlihat tidak membuat peluang menjadi 0.
- `alpha` adalah **hyperparameter** (kita yang menentukan).

Ingat: Naive Bayes **tidak memakai epoch atau gradient descent**. Saat `fit()`, model hanya **menghitung** peluang dari data latih, sekali jalan.

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X_train_vec, y_train)

print("Model Naive Bayes selesai dilatih!")
print("Kelas yang dikenali:", model.classes_)

# Prior: peluang awal tiap kelas (sebelum melihat isi SMS)
prior = np.exp(model.class_log_prior_)
for kelas, p in zip(model.classes_, prior):
    print(f"P({kelas}) = {p:.3f}")

## 8. Evaluasi Model
Sekarang model menebak **data uji** yang belum pernah dilihatnya, lalu kita bandingkan dengan jawaban yang benar.

In [ ]:
y_pred = model.predict(X_test_vec)

hasil = pd.DataFrame({"SMS": X_test.values, "Label Asli": y_test.values, "Tebakan Model": y_pred})
hasil["Benar?"] = np.where(hasil["Label Asli"] == hasil["Tebakan Model"], "✅", "❌")

print(f"Akurasi: {accuracy_score(y_test, y_pred) * 100:.1f}%\n")
hasil

### Confusion Matrix
Tabel yang menunjukkan **di mana model benar dan di mana model salah**:
- **Diagonal** (kiri atas → kanan bawah) = tebakan **benar**
- Di luar diagonal = tebakan **salah**

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=model.classes_, yticklabels=model.classes_,
            annot_kws={"size": 16})
plt.title("Confusion Matrix", fontweight="bold")
plt.xlabel("Tebakan Model")
plt.ylabel("Label Asli")
plt.show()

print(classification_report(y_test, y_pred))

**Cara membaca `classification_report`:**
- **precision**: dari semua yang ditebak spam, berapa persen yang benar-benar spam?
- **recall**: dari semua spam yang ada, berapa persen yang berhasil ditemukan?
- **f1-score**: gabungan precision dan recall
- **support**: jumlah data asli di kelas tersebut

## 9. Melihat Apa yang Dipelajari Model
Naive Bayes mudah "dibuka isinya". Kita bisa melihat peluang setiap kata, yaitu $P(\text{kata} \mid \text{kelas})$.

Kata yang **jauh lebih sering** muncul di spam dibanding di SMS normal adalah **ciri khas spam**.

In [ ]:
kata = vectorizer.get_feature_names_out()
idx_spam   = list(model.classes_).index("spam")
idx_normal = list(model.classes_).index("bukan spam")

# Peluang kata di tiap kelas: P(kata | kelas)
p_kata_spam   = np.exp(model.feature_log_prob_[idx_spam])
p_kata_normal = np.exp(model.feature_log_prob_[idx_normal])

df_kata = pd.DataFrame({
    "kata": kata,
    "P(kata|spam)": p_kata_spam,
    "P(kata|bukan spam)": p_kata_normal,
})
# Rasio > 1 artinya kata lebih condong ke spam
df_kata["rasio spam"] = df_kata["P(kata|spam)"] / df_kata["P(kata|bukan spam)"]

ciri_spam   = df_kata.sort_values("rasio spam", ascending=False).head(10)
ciri_normal = df_kata.sort_values("rasio spam").head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(ciri_spam["kata"][::-1], ciri_spam["rasio spam"][::-1], color="#e74c3c")
axes[0].set_title("Kata Paling Menunjukkan SPAM", fontweight="bold")
axes[0].set_xlabel("berapa kali lebih mungkin muncul di spam")

axes[1].barh(ciri_normal["kata"][::-1], (1 / ciri_normal["rasio spam"])[::-1], color="#2ecc71")
axes[1].set_title("Kata Paling Menunjukkan BUKAN SPAM", fontweight="bold")
axes[1].set_xlabel("berapa kali lebih mungkin muncul di bukan spam")

plt.tight_layout()
plt.show()

## 10. Coba dengan SMS Baru
Sekarang kita bisa menguji model dengan SMS **buatan sendiri**. Silakan ubah atau tambah isi daftar `sms_baru`!

Model juga memberi **peluang (%)** untuk setiap kelas lewat `predict_proba()`.

In [ ]:
sms_baru = [
    "selamat anda menang undian pulsa gratis",
    "besok jadi kerja kelompok di kampus",
    "klik link ini untuk klaim bonus saldo",
    "tolong kirim catatan kuliah tadi pagi",
    "promo diskon kuota internet hari ini",
]

def tebak_sms(daftar_sms):
    bersih = [bersihkan_teks(s) for s in daftar_sms]
    vektor = vectorizer.transform(bersih)
    tebakan = model.predict(vektor)
    peluang = model.predict_proba(vektor)

    for sms, label, p in zip(daftar_sms, tebakan, peluang):
        p_spam = p[idx_spam] * 100
        ikon = "🚨" if label == "spam" else "✅"
        print(f"{ikon} {label.upper():<11} (spam {p_spam:5.1f}%) | {sms}")

tebak_sms(sms_baru)

In [ ]:
# Coba ketik SMS kamu sendiri di sini
tebak_sms(["ketik sms kamu di sini"])

## 11. Eksperimen Hyperparameter `alpha`
`alpha` mengatur **besar smoothing**:
- `alpha` kecil (mendekati 0) → model sangat percaya pada data latih, tetapi kata yang belum pernah terlihat bisa membuat peluang hampir nol
- `alpha` besar → peluang tiap kata dibuat makin rata, sehingga model jadi "ragu-ragu"

Kita coba beberapa nilai `alpha` dan lihat pengaruhnya ke akurasi data uji.

In [ ]:
daftar_alpha = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
akurasi = []

for a in daftar_alpha:
    m = MultinomialNB(alpha=a).fit(X_train_vec, y_train)
    akurasi.append(accuracy_score(y_test, m.predict(X_test_vec)) * 100)

for a, acc in zip(daftar_alpha, akurasi):
    print(f"alpha = {a:<5} → akurasi {acc:.1f}%")

plt.figure(figsize=(8, 4))
plt.plot([str(a) for a in daftar_alpha], akurasi, marker="o", color="#3498db", linewidth=2)
plt.title("Pengaruh alpha terhadap Akurasi", fontweight="bold")
plt.xlabel("alpha")
plt.ylabel("Akurasi data uji (%)")
plt.ylim(0, 105)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

> **Catatan:** data kita sangat kecil (50 SMS, data uji hanya 13), jadi perbedaan akurasinya bisa kecil atau bahkan sama.
> Pada data sungguhan yang berisi ribuan SMS, pengaruh `alpha` akan lebih terasa.

## Rangkuman

| Tahap ML | Yang Kita Lakukan |
|----------|-------------------|
| **Data** | 50 SMS berlabel `spam` / `bukan spam` |
| **Feature Engineering** | Membersihkan teks + `CountVectorizer` (teks → jumlah kata) |
| **Algoritma** | `MultinomialNB` |
| **Parameter** (dipelajari mesin) | Prior $P(\text{kelas})$ dan $P(\text{kata} \mid \text{kelas})$ |
| **Hyperparameter** (diatur kita) | `alpha` (Laplace smoothing) |
| **Evaluasi** | Akurasi, confusion matrix, precision, recall |

**Poin penting:**
1. Naive Bayes menebak kelas dengan **membandingkan peluang**.
2. Model belajar **kata apa yang khas** untuk setiap kelas (lihat grafik di bagian 9).
3. Training-nya **sangat cepat** karena hanya menghitung, tanpa epoch atau gradient descent.
4. Selalu uji model dengan **data uji** yang tidak dipakai saat belajar.

**Tantangan untuk dicoba:**
- Tambahkan SMS baru ke data latih, lalu lihat apakah akurasinya berubah.
- Ganti `CountVectorizer` dengan `TfidfVectorizer` (materi TF-IDF).
- Tambahkan kelas ketiga, misalnya `promo resmi` (dari operator) vs `penipuan`.